# Распарсим ссылки на другие документы внутри 845-П

In [46]:
import csv
import xml.etree.ElementTree as ET
import zipfile
from urllib.parse import urlparse

import pandas as pd
import requests
from tqdm.notebook import tqdm

# Пространства имён ODF и XLink
NS = {
    'text':  'urn:oasis:names:tc:opendocument:xmlns:text:1.0',
    'xlink': 'http://www.w3.org/1999/xlink',
}

def extract_links(odt_path, external_only=True):
    """
    Извлекает все гиперссылки из ODT-файла.

    :param odt_path: путь к .odt
    :param external_only: если True — только внешние ссылки (http/https),
                          иначе включая внутренние (#anchor) и mailto.
    :return: список словарей {'url': ..., 'text': ...}
    """
    links = []
    with zipfile.ZipFile(odt_path, 'r') as z, z.open('content.xml') as f:
        tree = ET.parse(f)
        root = tree.getroot()

        # .//text:a — все элементы <text:a> на любой глубине
        for a in root.findall('.//text:a', NS):
            href = a.get('{http://www.w3.org/1999/xlink}href')
            if not href:
                continue

            # Собираем ВЕСЬ текст внутри <text:a>, включая вложенные span'ы
            text = ''.join(a.itertext()).strip()

            if external_only:
                scheme = urlparse(href).scheme
                if scheme not in ('http', 'https'):
                    continue

            links.append({'url': href, 'text': text})
    return links


def dedupe(links):
    """Убирает дубликаты по URL, сохраняя первое вхождение."""
    seen, result = set(), []
    for ln in links:
        if ln['url'] not in seen:
            seen.add(ln['url'])
            result.append(ln)
    return result

In [30]:
path = "../data/raw/845_p.odt"
links = extract_links(path)

for i, ln in enumerate(links, 1):
    print(f"{i}. {ln['url']}\n   текст: {ln['text']}\n")
    # Убираем указание параграфа, чтобы сохранять только адрес документа
    # ln['url'] = ln['url'][:ln['url'].rfind("/")] 
    # Только айдишник документа в Гарант
    ln['url'] = ln['url'].split("/")[-2]

# Сохранение в CSV (удобно открыть в Excel)
with open('../data/raw/links.csv', 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['url', 'text'], delimiter=';')
    w.writeheader()
    w.writerows(links)
print(f"Всего уникальных ссылок: {len(links)} → сохранено в links.csv")

1. http://ivo.garant.ru/document/redirect/411309161/0
   текст: Положение Банка России от 2 ноября 2024 г. N 845-П
"О порядке расчета величины кредитного риска банками с применением банковских методик управления кредитным риском и моделей количественной оценки кредитного риска"

2. http://ivo.garant.ru/document/redirect/12127405/7204
   текст: части четвертой статьи 72

3. http://ivo.garant.ru/document/redirect/12127405/721
   текст: частей первой

4. http://ivo.garant.ru/document/redirect/12127405/7214
   текст: четвертой

5. http://ivo.garant.ru/document/redirect/12127405/7216
   текст: шестой

6. http://ivo.garant.ru/document/redirect/12127405/72111
   текст: одиннадцатой статьи 72 1

7. http://ivo.garant.ru/document/redirect/12127405/7212
   текст: частью второй

8. http://ivo.garant.ru/document/redirect/12127405/72111
   текст: частью одиннадцатой

9. http://ivo.garant.ru/document/redirect/12127405/721
   текст: статьи 72 1

10. http://ivo.garant.ru/document/redirect/12127405/623


# Прочитаем айдишники документов и выгрузим документы из Гарант

In [47]:
links = pd.read_csv("../data/raw/links.csv", sep = ";")

In [48]:
document_ids = list(links["url"].unique())

In [49]:
def download_odt(topic_id, token):
    # Строго фиксированный URL по спецификации Гарант API 2.3.0
    url = f"https://api.garant.ru/v2/topic/{topic_id}/download-odt" # html
    
    headers = {
        "Authorization": f"Bearer {token}"
    }
    
    print(f"Отправка запроса на URL: {url}") # Для самопроверки в консоли
    
    try:
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status()
        
        filename = f"../data/raw/document_{topic_id}.odt"
        with open(filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                
        print(f"Успешно скачано! Файл: {filename}")
        
    except requests.exceptions.HTTPError as e:
        print(f"Ошибка сервера Гарант: {e} (Код: {response.status_code})")
        if response.status_code == 401:
            print("Проверьте ваш токен авторизации.")
        elif response.status_code == 404:
            print(f"Документ с ID {topic_id} не найден в системе.")
    except Exception as e:
        print(f"Сетевая или системная ошибка: {e}")

# --- ЗАПУСК ---
# Замените только значения внутри кавычек
MY_TOKEN = "4fba8a34ce1b11f1b77a0050568d72f0" # Перенесите сюда ваш токен

In [50]:
for id in tqdm(document_ids):
    id = str(id)
    print(id)
    download_odt(topic_id=id, token=MY_TOKEN)
    break

  0%|          | 0/92 [00:00<?, ?it/s]

411309161
Отправка запроса на URL: https://api.garant.ru/v2/topic/411309161/download-odt
Ошибка сервера Гарант: 423 Client Error:  for url: https://api.garant.ru/v2/topic/411309161/download-odt (Код: 423)


In [27]:
from pathlib import Path

# Все объекты в папке (файлы и папки)
p = Path('../data/raw')
files = [str(item) for item in p.iterdir() if item.is_file()]

In [35]:
downloaded_ids = [s[s.find("_") + 1:s.rfind(".")] for s in files]

In [ ]:
to_download = list(set(document_ids) - set(downloaded_ids))